# init

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/Colab Notebooks/rlhf/trpo
%pwd

/content/drive/MyDrive/Colab Notebooks/rlhf/trpo


'/content/drive/MyDrive/Colab Notebooks/rlhf/trpo'

In [ ]:
# !git log --oneline # 0d4deaf (HEAD -> main, origin/main, origin/HEAD) atari demo videos functionality added

# Atari Qbert policy video demo

This notebook loads a trained TRPO/PPO checkpoint, runs inference-only Qbert episodes, records the best rollout, and writes an MP4 plus summary files.

**Deterministic** means `argmax_a pi(a|s)` at every step. **Stochastic** means sampling `a ~ pi(.|s)` at every step. Both are real single trajectories; no rewind or branching happens inside an episode.

In [3]:
%pwd

'/content/drive/MyDrive/Colab Notebooks/rlhf/trpo'

In [4]:
# Colab setup. Run this once in a fresh runtime.
# If the repo is already installed and Atari ROMs are already available, you can skip this cell.
!pip -q install -e . 'gymnasium[atari,accept-rom-license]>=0.29' 'autorom[accept-rom-license]>=0.6' 'imageio[ffmpeg]>=2.34' 'Pillow>=10'
!AutoROM --accept-license >/dev/null 2>&1 || true


  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.7/434.7 kB 12.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Building editable for trpo-repro (pyproject.toml) ... done


In [6]:
from trpo_repro.demos.atari_video import render_policy_video
from IPython.display import Video, display
from pathlib import Path

In [5]:
%ls configs/atari

beamrider_ppo_clip.yaml     qbert_ppo_clip_marathon.yaml
beamrider_single_path.yaml  qbert_ppo_clip.yaml
breakout_ppo_clip.yaml      qbert_single_path.yaml
breakout_single_path.yaml   seaquest_ppo_clip.yaml
enduro_ppo_clip.yaml        seaquest_ppo_kl_penalty.yaml
enduro_single_path.yaml     seaquest_single_path.yaml
pong_ppo_clip.yaml          spaceinvaders_ppo_clip.yaml
pong_random.yaml            spaceinvaders_single_path.yaml
pong_single_path.yaml


In [7]:
%ls outputs/qbert_*

outputs/qbert_ppo_clip:
seed_0/

outputs/qbert_ppo_clip_marathon:
seed_0/

outputs/qbert_single_path:
seed_0/


In [ ]:
%ls outputs/qbert_single_path/seed_0

checkpoints/          git_diff.patch        run_manager_summary.json
config_resolved.yaml  launch_metadata.json  run_metadata.json
config_runtime.yaml   metrics.csv           run_summary.json
environment.json      metrics.jsonl


In [ ]:
%ls outputs/qbert_single_path/seed_0/checkpoints

best_checkpoint_info.json  epoch_0075.pt  epoch_0175.pt  epoch_0275.pt
best.pt                    epoch_0100.pt  epoch_0200.pt  epoch_0300.pt
epoch_0025.pt              epoch_0125.pt  epoch_0225.pt  last.pt
epoch_0050.pt              epoch_0150.pt  epoch_0250.pt


# stochastic run using trpo single path

In [ ]:

# Swap these two paths to render PPO instead of TRPO.
CONFIG_PATH = 'configs/atari/qbert_single_path.yaml'
CHECKPOINT_PATH = 'outputs/qbert_single_path/seed_0/checkpoints/best.pt'

# PPO alternative:
# CONFIG_PATH = 'configs/atari/qbert_ppo_clip.yaml'
# CHECKPOINT_PATH = '/content/trpo_runs/qbert_ppo_clip/seed_0/checkpoints/best.pt'

OUTPUT_DIR = 'outputs/videos/qbert_demo'
POLICY_MODE = 'stochastic'  # or 'stochastic'
DEVICE = 'cuda'  # use 'cpu' if CUDA is unavailable


In [ ]:

summary = render_policy_video(
    config_path=CONFIG_PATH,
    checkpoint_path=CHECKPOINT_PATH,
    output_dir=OUTPUT_DIR,
    device=DEVICE,
    seed=0,
    episodes=5,
    policy_mode=POLICY_MODE,
    record_selection='best',
    fps=30,
    scale=3,
    title='TRPO/PPO Atari Qbert Agent',
)
summary


VideoRunSummary(config_path='configs/atari/qbert_single_path.yaml', checkpoint_path='outputs/qbert_single_path/seed_0/checkpoints/best.pt', output_dir='outputs/videos/qbert_demo', policy_mode='stochastic', episodes=5, selected_episode=2, selected_return=1300.0, selected_length=576, return_mean=825.0, return_std=250.49950099750697, length_mean=493.0, video_path='outputs/videos/qbert_demo/qbert_v5_trpo_trpo_paper_stochastic_ep2.mp4', json_path='outputs/videos/qbert_demo/video_summary.json', csv_path='outputs/videos/qbert_demo/episode_summary.csv')

In [ ]:

display(Video(summary.video_path, embed=True, html_attributes='controls loop'))
print('Video:', summary.video_path)
print('JSON summary:', summary.json_path)
print('CSV summary:', summary.csv_path)


Video: outputs/videos/qbert_demo/qbert_v5_trpo_trpo_paper_stochastic_ep2.mp4
JSON summary: outputs/videos/qbert_demo/video_summary.json
CSV summary: outputs/videos/qbert_demo/episode_summary.csv


# deterministic run using trpo single path

In [ ]:
from IPython.display import Video, display

display(Video(summary.video_path, embed=True, html_attributes='controls loop'))
print('Video:', summary.video_path)
print('JSON summary:', summary.json_path)
print('CSV summary:', summary.csv_path)


Video: outputs/videos/qbert_demo/qbert_v5_trpo_trpo_paper_deterministic_ep2.mp4
JSON summary: outputs/videos/qbert_demo/video_summary.json
CSV summary: outputs/videos/qbert_demo/episode_summary.csv


# the qbert ppo marathon run

In [9]:

# Swap these two paths to render PPO instead of TRPO.
CONFIG_PATH = 'configs/atari/qbert_ppo_clip_marathon.yaml'
CHECKPOINT_PATH = 'outputs/qbert_ppo_clip_marathon/seed_0/checkpoints/best.pt'

# PPO alternative:
# CONFIG_PATH = 'configs/atari/qbert_ppo_clip.yaml'
# CHECKPOINT_PATH = '/content/trpo_runs/qbert_ppo_clip/seed_0/checkpoints/best.pt'

OUTPUT_DIR = 'outputs/videos/qbert_demo'
POLICY_MODE = 'deterministic'  # or 'stochastic'
DEVICE = 'cuda'  # use 'cpu' if CUDA is unavailable


In [10]:

summary = render_policy_video(
    config_path=CONFIG_PATH,
    checkpoint_path=CHECKPOINT_PATH,
    #output_dir=OUTPUT_DIR,
    device=DEVICE,
    seed=0,
    episodes=5,
    policy_mode=POLICY_MODE,
    record_selection='best',
    fps=30,
    scale=3,
    title='PPO Atari Qbert Agent trained on 2000 epochs',
)
summary


VideoRunSummary(config_path='configs/atari/qbert_ppo_clip_marathon.yaml', checkpoint_path='outputs/qbert_ppo_clip_marathon/seed_0/checkpoints/best.pt', output_dir='outputs/videos/qbert_demo', run_name='qbert_ppo_clip_marathon', environment='ALE/Qbert-v5', method='ppo', variant='clip', checkpoint_name='best.pt', policy_mode='deterministic', record_selection='best', seed=0, episodes=5, selected_episode=1, selected_return=15275.0, selected_length=1701, return_mean=14500.0, return_std=1452.0674915443842, length_mean=1548.6, artifact_stem='qbert_ppo_clip_marathon__ppo-clip__ckpt-best__deterministic__seed-0__best-of-5', video_path='outputs/videos/qbert_demo/qbert_ppo_clip_marathon__ppo-clip__ckpt-best__deterministic__seed-0__best-of-5__selected-ep-1.mp4', json_path='outputs/videos/qbert_demo/qbert_ppo_clip_marathon__ppo-clip__ckpt-best__deterministic__seed-0__best-of-5__summary.json', csv_path='outputs/videos/qbert_demo/qbert_ppo_clip_marathon__ppo-clip__ckpt-best__deterministic__seed-0__bes

In [11]:
display(Video(summary.video_path, embed=True, html_attributes='controls loop'))
print('Video:', summary.video_path)
print('JSON summary:', summary.json_path)
print('CSV summary:', summary.csv_path)


Output hidden; open in https://colab.research.google.com to view.